In [ ]:
import os
import time
import requests
import pandas as pd
import json
#------------OLD METHOD--------------------------
# PLEASE DO NOT SHARE KEY
# API_KEY = "redacted" 
#-----------------------------------------------------------------
# API key has been moved to retrieval_config.json file, This file is no longer tracked by git. Please generate your own API key at https://www.eia.gov/opendata/register.php and save it in retrieval_config.json if you want to use this code.

with open("retrieval_config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
#-------------------------------------------------------

def fetch_eia_v2_all(url: str, params: dict, *, page_size: int = 5000, max_pages: int = 500, sleep_s: float = 0.1):
    session = requests.Session()
    rows_all = []
    offset = int(params.get("offset", 0))

    for page in range(1, max_pages + 1):
        p = dict(params)
        p["api_key"] = API_KEY
        p["offset"] = offset
        p["length"] = page_size

        r = session.get(url, params=p, timeout=60)
        r.raise_for_status()
        payload = r.json()

        resp = payload.get("response", {})
        rows = resp.get("data", [])
        total = resp.get("total", None)

        if not rows:
            break

        rows_all.extend(rows)
        offset += len(rows)
        
        if len(rows) < page_size:
            break
        if isinstance(total, int) and offset >= total:
            break

        if sleep_s:
            time.sleep(sleep_s)

    return rows_all


In [ ]:
BASE_URL = "https://api.eia.gov/v2/electricity/retail-sales/data/"
sectors = ["ALL", "COM", "IND", "OTH", "RES", "TRA"]

params = {
    "frequency": "monthly",
    "data[0]": "sales",
    "start": "2001-01",
    "end": "2025-11",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000,
}

params["facets[sectorid][]"] = sectors

rows = fetch_eia_v2_all(BASE_URL, params, page_size=5000)
df_raw = pd.DataFrame(rows)

df = df_raw.copy()

df["period"] = pd.to_datetime(df["period"], format="%Y-%m", errors="coerce")
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")

cols = ["period", "stateid", "sectorid", "sales"]
if "stateDescription" in df.columns:
    cols.append("stateDescription")

df = df[cols].dropna(subset=["period", "stateid", "sectorid", "sales"])

df_pivot = (
    df.pivot_table(
        index=["period", "stateid"],
        columns="sectorid",
        values="sales",
        aggfunc="sum"   
    )
    .reindex(columns=sectors)  
    .sort_index(level=["period", "stateid"], ascending=[False, True])  
)

if "stateDescription" in df.columns:
    state_lookup = (
        df[["stateid", "stateDescription"]]
        .drop_duplicates("stateid")
        .set_index("stateid")
    )
    df_final = df_pivot.reset_index()
    df_final["stateDescription"] = df_final["stateid"].map(state_lookup["stateDescription"])   
    df_final = df_final[["period", "stateid", "stateDescription"] + sectors]

else:
    df_final = df_pivot.reset_index()

df_final.head()


In [ ]:
sorted(df_final["stateDescription"].unique())

In [ ]:
df_raw.shape

In [ ]:
df_final["salesUnit"] = "million kilowatt hours" 
df_final.head()

In [ ]:
#Output to parquet file type
df_final.to_parquet("data/eia_retail_sales_mwh_monthly_state_sectorwide.parquet", index=False)

In [ ]:
#Output to csv file type
df_final.to_csv("data/eia_retail_sales_mwh_monthly_state_sectorwide.csv", index=False)